# Van Hateren：复现 CV 驻点公式与边界外推

Run All 只读取小型谱缓存，重新求解析 stationary κ，并检查公式；不读取原图、不重跑大型 MC。可修改 n、sigma 网格，输出另存。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'scripts/pixel_cv_stationary.py').exists())
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'notebooks/outputs/pixel_ridge/vanhateren_selection_estimation'
OUT = DATA / 'notebook_reproduction'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42,
                     'font.family': 'DejaVu Sans', 'mathtext.fontset': 'dejavusans'})
print('Cache:', DATA)
print('New outputs:', OUT)


## 1. 实验谱与 teacher

s 是像素协方差特征值，beta 是圆盘 teacher 在该 eigenbasis 的坐标；没有 whitening 或截断。

In [ ]:
with np.load(ROOT / 'tables/vanhateren_disk_teacher_spectrum.npz') as cache:
    s = cache['eigenvalues'].astype(float)
    beta = cache['beta_proj'].astype(float)
S = float(s @ beta**2)
n = 1000
sigmas = np.sort(pd.read_csv(DATA / 'comparison_extended.csv').sigma.unique())
print(f'd={len(s)}, n={n}, S={S:.8f}, sigma range={sigmas[[0,-1]]}')


## 2. 解析条件

记 $B_{p,q}=\sum_j\beta_j^2s_j^p/(s_j+\kappa)^q$，$\mathrm{df}_{p,q}=\sum_js_j^p/(s_j+\kappa)^q$，$\mathrm{df}_2=\mathrm{df}_{2,2}$。

内部驻点满足
\[
(n-\mathrm{df}_2)\kappa B_{2,3}
=\mathrm{df}_{2,3}(\kappa^2B_{1,2}+\sigma^2).
\]
物理分支要求 $\lambda=\kappa(1-\mathrm{df}_1/n)\geq0$。在当前 d>n 的问题中下界 κ₀>0。

求根器寻找扫描区间内的局部极小值，再比较风险及边界；不是把不可行的根裁剪成内部解。

In [ ]:
import time
from scripts.pixel_cv_stationary import moments, stationary_table, extrapolate_at_boundary
start = time.perf_counter()
stationary = stationary_table(s, beta, n, sigmas)
extended = extrapolate_at_boundary(stationary, s, beta)
print(f'Elapsed: {time.perf_counter()-start:.3f}s')
display(stationary[['sigma','status','kappa','lam','E_gen','z_acc']])


## 3. 独立计算任意 κ 的真实 leading-DE 比值

\[
E_{\rm gen}=\frac{n(\kappa^2B_{1,2}+\sigma^2)}{n-\mathrm{df}_2}-\sigma^2,\quad
\mu_N=B_{1,1},\quad
\mu_D=B_{2,2}+\frac{\mathrm{df}_{1,2}}{n}(E_{\rm gen}+\sigma^2).
\]

论文的简化式仅在驻点残差为零时等于 $\mu_D/\mu_N-1$：
\[
z_{\rm paper}=\frac{\kappa\mathrm{df}_{1,2}}{B_{1,1}}
\left(\frac{B_{2,3}}{\mathrm{df}_{2,3}}-\frac{B_{1,2}}{\mathrm{df}_{1,2}}\right).
\]
下面保留低噪声 κ₀ 直接代入，但同时画正确的 boundary leading-DE。

In [ ]:
rows = []
for row in extended.itertuples():
    k, sigma = row.kappa, row.sigma
    b11,b12,b23,d12,d23,d2,h = moments(s,beta,k,n)
    egen = n/(n-d2)*(k*k*b12+sigma*sigma)-sigma*sigma
    muD = np.sum(beta**2*(s/(s+k))**2) + d12/n*(egen+sigma*sigma)
    z = muD/b11-1
    correction = d12/((n-d2)*b11)*(sigma*sigma-h)
    np.testing.assert_allclose(z, row.z_acc+correction, atol=1e-10)
    rows.append(dict(sigma=sigma, kappa=k, z_direct=z,
                     z_paper=row.z_acc, residual_correction=correction,
                     E_gen=egen/S, E_acc=(z/(1+z))**2,
                     R2_acc=1-z*z, slope_acc=1/(1+z)))
direct = pd.DataFrame(rows)
mask = ~extended.extrapolated
np.testing.assert_allclose(direct.loc[mask,'z_direct'],
                           extended.loc[mask,'z_acc'], atol=1e-10)
display(direct.head(8))
print('Interior identity and residual correction verified.')


## 4. 比较三个 accentuation 指标

虚线：论文公式直接代入；实线：同一个 κ 上完整 leading-DE。低噪声二者不同正是驻点条件不成立的结果，不是 MC fluctuation。

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax, metric, label in zip(axes, ['E_acc','R2_acc','slope_acc'],
                             [r'$E_{acc}/S$', r'$R^2_{acc}$', r'$slope_{acc}$']):
    ax.plot(sigmas, direct[metric], label='Full leading DE at same kappa')
    ax.plot(sigmas, extended[metric], '--', label='Paper formula substitution')
    boundary = extended.extrapolated
    ax.scatter(sigmas[boundary], extended.loc[boundary,metric],
               facecolors='none',edgecolors='tab:orange',marker='s')
    ax.set_xscale('symlog',linthresh=0.01)
    ax.set_xlabel(r'Response noise $\sigma$')
    ax.set_ylabel(label)
    ax.grid(alpha=.2)
    top = ax.secondary_xaxis('top',functions=(lambda x:x*x/S,
                                              lambda r:np.sqrt(np.maximum(r,0)*S)))
    top.set_xlabel(r'Noise / signal variance $\sigma^2/S$')
axes[0].set_yscale('symlog', linthresh=1e-10)
axes[1].set_yscale('symlog', linthresh=1)
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
for name, frame in [('stationary_recomputed',stationary),
                    ('stationary_extrapolated_recomputed',extended),
                    ('stationary_direct_comparison',direct)]:
    frame.to_csv(OUT / f'{name}.csv',index=False)
for ext in ('png','pdf'):
    fig.savefig(OUT / f'stationary_boundary_check.{ext}',dpi=180,bbox_inches='tight')
# Verify reproduction of existing cache only for the original n/grid.
if n == 1000:
    saved = pd.read_csv(DATA / 'stationary_cv.csv')
    if np.array_equal(sigmas,saved.sigma.to_numpy()):
        for col in ['kappa','lam','z_acc','E_acc','R2_acc','slope_acc']:
            np.testing.assert_allclose(stationary[col],saved[col],
                                       rtol=1e-7,atol=1e-10,equal_nan=True)
        print('Recomputed stationary table matches existing cache.')
print('Saved to:',OUT)


## 修改实验

修改 n 或 sigmas 后，从第 2 节继续运行。与自然图像 MC 的对应关系只在原始 n=1000、原始谱和噪声网格下成立；改变这些参数不会自动生成新的 MC。

要将重新计算的结果叠加到完整三列图，可把本 notebook 的 stationary 或 extended 传给第一个 notebook 使用的 plot(..., stationary=...)。